# Spec section 7: segmentation and export with `tpt_bio`

This notebook follows the segmentation and export steps of spec section 7 (High-Content Phenotypic Screening) using only the bindings in `tpt_bio`.

| Spec step | Status in the bindings |
|---|---|
| Ingestion (OME-TIFF from S3) | PNG and TIFF are bound (`load_image`). This notebook builds a synthetic image instead, and OME-TIFF from S3 is not bound. |
| Preprocessing (3D Gaussian, background subtraction) | Not bound. |
| Segmentation (marker-controlled watershed) | Bound: `otsu_threshold`, `apply_threshold`, `label_components`, `watershed_2d`. |
| Feature extraction | Bound: `object_features` (shape and intensity columns). |
| Statistics (PCA, clustering) | Not bound in Python. Use the pipeline crate, or NumPy. |
| Export to Jupyter | Bound: results are NumPy arrays. |

Install with `pip install tpt-bio numpy`.

In [1]:
import numpy as np
import tpt_bio

print("tpt_bio", tpt_bio.__version__)

tpt_bio 0.1.0


## Synthetic input

Two touching blobs stand in for a cell image. The array is `(rows, cols)`.

In [2]:
yy, xx = np.mgrid[0:40, 0:40]
img = (
    255 * np.exp(-((xx - 12) ** 2 + (yy - 20) ** 2) / (2 * 36.0))
    + 255 * np.exp(-((xx - 27) ** 2 + (yy - 20) ** 2) / (2 * 36.0))
)
img.shape, float(img.max())

((40, 40), 268.2435895669139)

## Threshold and label

Otsu picks the foreground cut-off. Labelling uses 8-connectivity.

In [3]:
t = tpt_bio.otsu_threshold(img)
mask = tpt_bio.apply_threshold(img, t)
labels, count = tpt_bio.label_components(mask, 8)
print(f"threshold={t:.2f} foreground={int(mask.sum())} components={count}")

threshold=106.96 foreground=430 components=1


## Marker-controlled watershed

One seed per cell centre separates the touching blobs. Flooding is restricted to the foreground mask.

In [4]:
markers = np.zeros(img.shape, dtype=np.uint32)
markers[20, 12] = 1
markers[20, 27] = 2

# Flood from the bright centres: watershed treats low values as basins.
cells = tpt_bio.watershed_2d(-img, markers, mask)
print("labels present:", np.unique(cells).tolist())

labels present: [0, 1, 2]


## Per-object summary

Per-object features come from `object_features`: one list per column, one value per cell.

In [5]:
feats = tpt_bio.object_features(cells, img)
for label, area, mean in zip(feats["label"], feats["area"], feats["mean"]):
    print(f"cell {int(label)}: {int(area)} px, mean intensity {mean:.1f}")

cell 1: 215 px, mean intensity 185.9
cell 2: 215 px, mean intensity 185.9
